In [1]:
# Standard libraries
import sys
# Add your custom path
gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)
import os
import logging
import argparse # Argument parsing

# Data manipulation and analysis
import pandas as pd
import numpy as np
import pickle
import torch
import torch.optim as optim
import copy                    # clone tensor
import time

# Custom imports

from GEMS_TCO import kernels_reparam_space_time_gpu as kernels_reparam_space_time
from GEMS_TCO import kernels_reparam_space_time_gpu_031026 as kernels_reparam_space_time_gpu_031026
from GEMS_TCO import orderings as _orderings 
from GEMS_TCO import alg_optimization, BaseLogger
from GEMS_TCO import kernels_columns as kernels_reparam_space_time_gpu_col
from typing import Optional, List, Tuple
from pathlib import Path
import typer
import json
from json import JSONEncoder
from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data2, exact_location_filter, load_data_dynamic_processed
from GEMS_TCO import debiased_whittle as debiased_whittle
from torch.nn import Parameter

Load daily data applying max-min ordering

In [2]:
space: List[str] = ['1', '1']
lat_lon_resolution = [int(s) for s in space]
mm_cond_number: int = 8
years = ['2024']
month_range = [7] 

output_path = input_path = Path(config.mac_estimates_day_path)
data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)
#data_load_instance = load_data2(config.mac_data_load_path)


#lat_range_input = [1, 3]
#lon_range_input = [125.0, 129.0]

lat_range_input=[-3,2]      
lon_range_input=[121, 131] 
#lat_range_input=[-3,-1]      
#lon_range_input=[121, 125] 

df_map, ord_mm, nns_map, day_offsets = data_load_instance.load_maxmin_ordered_data_bymonthyear(
lat_lon_resolution=lat_lon_resolution, 
mm_cond_number=mm_cond_number,
years_=years, 
months_=month_range,

lat_range=lat_range_input,   
lon_range=lon_range_input,
is_whittle= False
  
)

--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---


In [6]:
daily_aggregated_tensors_dw = [] 
daily_hourly_maps_dw = []      

daily_aggregated_tensors_vecc = [] 
daily_hourly_maps_vecc = []   

for day_index in range(31):
    hour_start_index = day_index * 8
    hour_end_index = (day_index + 1) * 8
    hour_indices = [hour_start_index, hour_end_index]

    # --- DW용 데이터 로드 (day_offsets 인자 추가) ---
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map, 
        day_offsets,  # <--- 이 부분이 추가되어야 합니다
        hour_indices, 
        ord_mm= None,  # DW에서는 ord_mm이 필요 없으므로 None으로 설정
        dtype=torch.float64, 
        keep_ori=False
    )
    daily_aggregated_tensors_dw.append(day_aggregated_tensor)
    daily_hourly_maps_dw.append(day_hourly_map)

    # --- Vecchia용 데이터 로드 (day_offsets 인자 추가) ---
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map, 
        day_offsets,  # <--- 이 부분이 추가되어야 합니다
        hour_indices, 
        ord_mm=ord_mm,
        dtype=torch.float64, 
        keep_ori= False
    )
    daily_aggregated_tensors_vecc.append(day_aggregated_tensor)
    daily_hourly_maps_vecc.append(day_hourly_map)

print(f"Aggregated Tensor Shape: {daily_aggregated_tensors_vecc[0].shape}")
# 예상 출력: torch.Size([행수, 12]) -> 열이 12개여야 성공입니다.

Aggregated Tensor Shape: torch.Size([145008, 11])


In [11]:
daily_hourly_maps_vecc[0]['2024_07_y24m07day01_hm00:53']

tensor([[ -0.4640, 126.0230,  -0.2126,  ...,   0.0000,   0.0000,   0.0000],
        [ -2.9720, 121.0460,   0.8263,  ...,   0.0000,   0.0000,   0.0000],
        [ -2.9720, 131.0000,      nan,  ...,   0.0000,   0.0000,   0.0000],
        ...,
        [ -0.5080, 127.6610,   0.1978,  ...,   0.0000,   0.0000,   0.0000],
        [ -0.4640, 122.9360,  -4.5268,  ...,   0.0000,   0.0000,   0.0000],
        [ -0.5080, 127.9130,   0.6522,  ...,   0.0000,   0.0000,   0.0000]],
       dtype=torch.float64)

In [8]:
v=0.5
mm_cond_number= 8
nheads = 300
patience, factor = 5, 0.5


In [9]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Global L-BFGS Settings
LBFGS_LR = 1.0
LBFGS_MAX_STEPS = 2      # 10 to 20  
LBFGS_HISTORY_SIZE = 100   
LBFGS_MAX_EVAL = 100       # line search from 50 to 80
       
DELTA_LAT, DELTA_LON = 0.044, 0.063 
LAT_COL, LON_COL, VAL_COL, TIME_COL = 0, 1, 2, 3

days_list = [20]

# --- 2. Run optimization loop over pre-loaded data ---

for day_idx in days_list:  # 0-based

    print(f'\n{"="*40}')
    print(f'--- Starting Processing for Day {day_idx+1} (2024-07-{day_idx+1}) ---')
    print(f'{"="*40}')

    # Assuming data access is correct
    daily_hourly_map_dw = daily_hourly_maps_dw[day_idx]
    daily_aggregated_tensor_dw = daily_aggregated_tensors_dw[day_idx]

    daily_hourly_map_vecc = daily_hourly_maps_vecc[day_idx]
    daily_aggregated_tensor_vecc = daily_aggregated_tensors_vecc[day_idx]

    if isinstance(daily_aggregated_tensor_vecc, torch.Tensor):
        daily_aggregated_tensor_vecc = daily_aggregated_tensor_vecc.to(DEVICE)


    init_sigmasq   = 13.059
    init_range_lat = 0.2
    init_range_lon = 0.25
    init_advec_lat = 0.0218
    init_range_time = 1.5
    init_advec_lon = -0.1689
    init_nugget    = 0.247

    #init_sigmasq   = 13.059
    #init_range_lat = 0.154 
    #init_range_lon = 0.195
    #init_advec_lat = 1e-8
    #init_range_time = 1e-8
    #init_advec_lon = 1e-8
    #init_nugget    = 0.247
    
    
    # Map model parameters to the 'phi' reparameterization
    init_phi2 = 1.0 / init_range_lon                # 1/range_lon
    init_phi1 = init_sigmasq * init_phi2            # sigmasq / range_lon
    init_phi3 = (init_range_lon / init_range_lat)**2  # (range_lon / range_lat)^2
    init_phi4 = (init_range_lon / init_range_time)**2      # (range_lon / range_time)^2

    # Create Initial Parameters (Float64, Requires Grad)
    initial_vals = [np.log(init_phi1), np.log(init_phi2), np.log(init_phi3), 
                    np.log(init_phi4), init_advec_lat, init_advec_lon, np.log(init_nugget)]

    params_list = [
        torch.tensor([val], requires_grad=True, dtype=torch.float64, device=DEVICE)
        for val in initial_vals
    ]

    # --- 💥 Instantiate the GPU Batched Class ---
    # NOTE: Ensure fit_vecchia_lbfgs is the NEW class we defined
    model_instance = kernels_reparam_space_time_gpu_031026.fit_vecchia_lbfgs(
            smooth = v,
            input_map = daily_hourly_map_vecc,
            aggregated_data = daily_aggregated_tensor_vecc,
            nns_map = nns_map,
            mm_cond_number = mm_cond_number,
            nheads = nheads
        )

    '''  
    model_instance = kernels_reparam_space_time_gpu_col.fit_vecchia_lbfgs(
        smooth=v,
        #input_map=daily_hourly_maps_vecc_sim[0],
        #aggregated_data= daily_aggregated_tensors_vecc_sim[0],

        input_map=daily_hourly_maps_dw[2],
        aggregated_data= daily_aggregated_tensors_dw[2],

        nns_map=None,
        mm_cond_number=mm_cond_number,
        #nheads=nheads
    )
    '''

    # --- 💥 Set L-BFGS Optimizer ---
    optimizer_vecc = model_instance.set_optimizer(
                params_list,     
                lr=LBFGS_LR,            
                max_iter=LBFGS_MAX_EVAL,        
                history_size=LBFGS_HISTORY_SIZE 
            )

    print(f"\n--- Starting Phase 2: Vecchia Optimization (Day {day_idx+1}) ---")
    start_time = time.time()
    
    # --- 💥 Call the Batched Fit Method ---
    # REMOVED: model_instance.matern_cov_aniso_STABLE_log_reparam
    out, steps_ran = model_instance.fit_vecc_lbfgs(
            params_list,
            optimizer_vecc,
            # covariance_function argument is GONE
            max_steps=LBFGS_MAX_STEPS, 
            grad_tol=1e-7
        )


    end_time = time.time()
    epoch_time = end_time - start_time
    
    print(f"Vecchia Optimization finished in {epoch_time:.2f}s. Results: {out}")


Using device: cpu

--- Starting Processing for Day 21 (2024-07-21) ---

--- Starting Phase 2: Vecchia Optimization (Day 21) ---
🚀 Pre-computing (Daily AR1 Strategy: A(8), B(9), C(9)) with NaN Filtering... [Mean Lat: -0.4796] ✅ Done. (Heads: 2367, Tails: 141549)
--- Starting Batched L-BFGS Optimization (GPU) ---
--- Step 1/2 / Loss: 1.268807 ---
  Param 0: Value=2.8872, Grad=9.528155778716066e-06
  Param 1: Value=0.8545, Grad=-3.3660873102758996e-06
  Param 2: Value=0.5971, Grad=1.234100802395804e-06
  Param 3: Value=-2.5040, Grad=7.847708418795015e-07
  Param 4: Value=0.0117, Grad=6.084466697288404e-06
  Param 5: Value=-0.1298, Grad=3.423029126963656e-06
  Param 6: Value=0.5811, Grad=6.850174428514903e-06
  Max Abs Grad: 9.528156e-06
------------------------------
--- Step 2/2 / Loss: 1.243727 ---
  Param 0: Value=2.8872, Grad=9.528155778716066e-06
  Param 1: Value=0.8545, Grad=-3.3660873102758996e-06
  Param 2: Value=0.5971, Grad=1.234100802395804e-06
  Param 3: Value=-2.5040, Grad=7.

Using device: cpu

========================================
--- Starting Processing for Day 21 (2024-07-21) ---
========================================

Final Interpretable Params: {'sigma_sq': 7.63455828686993, 'range_lon': 0.4254899509039601, 'range_lat': 0.31566923592786866, 'range_time': 1.488084006310298, 'advec_lat': 0.011693149833305456, 'advec_lon': -0.12980875225221483, 'nugget': 1.7879304175442052}
Vecchia Optimization finished in 121.35s. Results: [2.8871990316744807, 0.8545139483945773, 0.5970927738873585, -2.504006678131201, 0.011693149833305456, -0.12980875225221483, 0.5810587596317598, 1.2437272642248425]


--- Starting Phase 2: Vecchia Optimization (Day 21) ---
🚀 Pre-computing (Daily AR1 Strategy: A(8), B(9), C(9)) with NaN Filtering... [Mean Lat: -0.4797] ✅ Done. (Heads: 2367, Tails: 141549)
--- Starting Batched L-BFGS Optimization (GPU) ---
--- Step 1/2 / Loss: 1.267904 ---
  Param 0: Value=2.8958, Grad=4.6873516415997e-06
  Param 1: Value=0.8627, Grad=-4.548332428453381e-06
  Param 2: Value=0.5966, Grad=-1.4111176539332172e-06
  Param 3: Value=-2.5140, Grad=1.3976498191157534e-06
  Param 4: Value=0.0114, Grad=-8.997937196966038e-06
  Param 5: Value=-0.1356, Grad=2.221484381725869e-06
  Param 6: Value=0.5741, Grad=3.436997536843437e-06
  Max Abs Grad: 8.997937e-06
------------------------------
--- Step 2/2 / Loss: 1.243235 ---
  Param 0: Value=2.8958, Grad=4.6873516415997e-06
  Param 1: Value=0.8627, Grad=-4.548332428453381e-06
  Param 2: Value=0.5966, Grad=-1.4111176539332172e-06
  Param 3: Value=-2.5140, Grad=1.3976498191157534e-06
  Param 4: Value=0.0114, Grad=-8.997937196966038e-06
  Param 5: Value=-0.1356, Grad=2.221484381725869e-06
  Param 6: Value=0.5741, Grad=3.436997536843437e-06
  Max Abs Grad: 8.997937e-06
------------------------------
Final Interpretable Params: {'sigma_sq': 7.638148892072283, 'range_lon': 0.4220272559737539, 'range_lat': 0.3131704455246653, 'range_time': 1.4833890243595544, 'advec_lat': 0.011443792891355206, 'advec_lon': -0.13558416621224342, 'nugget': 1.775510538022371}
Vecchia Optimization finished in 134.88s. Results: [2.8958406616237835, 0.8626853794162849, 0.5966446056994693, -2.5140294614672114, 0.011443792891355206, -0.13558416621224342, 0.5740880086255047, 1.243235497705691]

In [ ]:
irr. (-3,2 121,131, intercept, time dummies, latitude)

Final Interpretable Params: {
    
'sigma_sq': 14.557378105972255, 
 'range_lon': 0.36297285215444397, 'range_lat': 0.33986461719454436, 'range_time': 1.8307357609214716, 'advec_lat': -0.0014072484629318405, 'advec_lon': -0.16623085410420968, 'nugget': 0.24999425486908594}
Vecchia Optimization finished in 67.75s. Results: [3.691525186381127, 1.0134272349673297, 0.13156138084623087, -3.2362903523329813, -0.0014072484629318405, -0.16623085410420968, -1.3863173419076031, 0.9065025151721705]

regular (hide nugget)
------------------------------
------------------------------
Final Interpretable Params: {'sigma_sq': 13.881823879707401, 'range_lon': 0.3308191657110315, 'range_lat': 0.3048282266752421, 'range_time': 1.6460269337530522, 'advec_lat': -0.0014028377727820455, 'advec_lon': -0.15541841240605433, 'nugget': 1.112667285815519e-07}
Vecchia Optimization finished in 141.61s. Results: [3.7367637299362526, 1.1061833800531369, 0.16364694440410635, -3.209095690658186, -0.0014028377727820455, -0.15541841240605433, -16.01133555793201, 0.9490311091657573]


irr  ar(1) offset logic 이용했을때 사실 차이 없고 latitude centering 했을 때 소수점 10쨰자리에서 살짝 개선



SyntaxError: invalid decimal literal (2978187764.py, line 7)

fit debiased whittle

In [ ]:
import time
import torch
import numpy as np
from torch.nn import Parameter

# Device 설정 (Whittle=CPU)
DEVICE_DW = torch.device("cpu")

# Global Constants
dwl = debiased_whittle.debiased_whittle_likelihood()
TAPERING_FUNC = dwl.cgn_hamming 
NUM_RUNS = 1
DWL_MAX_STEPS = 15       
LAT_COL, LON_COL, VAL_COL, TIME_COL = 0, 1, 2, 3

days_list = [10]  # 0-based index for days (0 to 30 for July)

# --- Main Loop ---
for day_idx in days_list:
    print(f'\n{"="*50}')
    print(f'--- Processing Real Data: Day {day_idx+1} (2024-07-{day_idx+1}) ---')
    print(f'{"="*50}')

    start_time = time.time()

    try:
        # Data Prepare
        daily_hourly_map_dw = daily_hourly_maps_dw[day_idx]
        daily_aggregated_tensor_dw = daily_aggregated_tensors_dw[day_idx].to(DEVICE_DW)

        if daily_aggregated_tensor_dw.shape[0] == 0:
            print(f"Skipping Day {day_idx+1}: No data.")
            continue

        # Initial Parameters (7-Parameter 설정)
        init_sigmasq   = 13.059
        init_range_lat = 0.154 
        init_range_lon = 0.195
        init_advec_lat = 0.0218
        init_range_time = 0.7  
        init_advec_lon = -0.1689
        init_nugget    = 0.247
        
        raw_init_floats = [init_sigmasq, init_range_lat, init_range_lon, init_range_time, 
                            init_advec_lat, init_advec_lon, init_nugget]

        # -------------------------------------------------------
        # STEP 1: Debiased Whittle Pre-process
        # -------------------------------------------------------
        db = debiased_whittle.debiased_whittle_preprocess(
            daily_aggregated_tensors_dw, daily_hourly_maps_dw, day_idx=day_idx, 
            params_list=raw_init_floats, lat_range=[-3, 2], lon_range=[121.0, 131.0]
        )

        cur_df = db.generate_spatially_filtered_days(-3, 2, 121, 131).to(DEVICE_DW)
        
        if cur_df.numel() == 0 or cur_df.shape[1] <= max(LAT_COL, LON_COL, VAL_COL, TIME_COL):
            print(f"Error: Data for Day {day_idx+1} is empty or invalid.")
            continue

        unique_times = torch.unique(cur_df[:, TIME_COL])
        time_slices_list = [cur_df[cur_df[:, TIME_COL] == t_val] for t_val in unique_times]

        # -------------------------------------------------------
        # STEP 2: Pre-compute J-vector, Taper Grid
        # -------------------------------------------------------
        print("\nPre-computing J-vector (Hamming taper)...")
        J_vec, n1, n2, p_time, taper_grid = dwl.generate_Jvector_tapered( 
            time_slices_list, tapering_func=TAPERING_FUNC, 
            lat_col=LAT_COL, lon_col=LON_COL, val_col=VAL_COL, device=DEVICE_DW
        )
        
        if J_vec is None or J_vec.numel() == 0 or n1 == 0 or n2 == 0 or p_time == 0:
           print(f"Error: J-vector generation failed for Day {day_idx+1}.")
           continue
           
        print("Pre-computing sample periodogram...")
        I_sample = dwl.calculate_sample_periodogram_vectorized(J_vec)

        print("Pre-computing Hamming taper autocorrelation...")
        taper_autocorr_grid = dwl.calculate_taper_autocorrelation_fft(taper_grid, n1, n2, DEVICE_DW)

        if torch.isnan(I_sample).any() or torch.isinf(I_sample).any():
            print("Error: NaN/Inf in sample periodogram.")
            continue
        if torch.isnan(taper_autocorr_grid).any() or torch.isinf(taper_autocorr_grid).any():
            print("Error: NaN/Inf in taper autocorrelation.")
            continue

        print(f"Data grid: {n1}x{n2}, {p_time} time points. J-vector, Periodogram, Taper Autocorr on {DEVICE_DW}.")

        # -------------------------------------------------------
        # STEP 3: Optimization Loop (NUM_RUNS)
        # -------------------------------------------------------
        all_final_results = []
        all_final_losses = []

        for i in range(NUM_RUNS):
            print(f"\n{'='*30} Initialization Run {i+1}/{NUM_RUNS} {'='*30}")
            
            init_phi2 = 1.0 / init_range_lon
            init_phi1 = init_sigmasq * init_phi2
            init_phi3 = (init_range_lon / init_range_lat)**2
            init_phi4 = (init_range_lon / init_range_time)**2

            initial_params_values = [
                np.log(init_phi1),
                np.log(init_phi2),
                np.log(init_phi3),
                np.log(init_phi4),
                init_advec_lat,
                init_advec_lon,
                np.log(init_nugget)
            ]
            
            print(f"Starting with FIXED params (raw log-scale): {[round(p, 4) for p in initial_params_values]}")

            params_list = [
                Parameter(torch.tensor([val], dtype=torch.float64, device=DEVICE_DW))
                for val in initial_params_values
            ]

            optimizer = torch.optim.LBFGS(
                params_list, lr=1.0, max_iter=DWL_MAX_STEPS, history_size=100, 
                line_search_fn="strong_wolfe", tolerance_grad=1e-5
            )

            print(f"Starting optimization run {i+1} on device {DEVICE_DW} (Hamming, 7-param ST kernel, L-BFGS)...")
            
            # --- 💥 REVISED: Call L-BFGS trainer, pass p_time 💥 ---
            nat_params_str, phi_params_str, raw_params_str, loss, steps_run = dwl.run_lbfgs_tapered(
                params_list=params_list,
                optimizer=optimizer,
                I_sample=I_sample,
                n1=n1, n2=n2, p_time=p_time,
                taper_autocorr_grid=taper_autocorr_grid, 
                max_steps=DWL_MAX_STEPS,
                device=DEVICE_DW
            )
            # --- END REVISION ---
            
            if loss is not None:
                all_final_results.append((nat_params_str, phi_params_str, raw_params_str))
                all_final_losses.append(loss)
            else:
                all_final_losses.append(float('inf'))

        # -------------------------------------------------------
        # STEP 4: Summary Output
        # -------------------------------------------------------
        print(f"\n\n{'='*25} Overall Result from Run {'='*25} {'='*25}")
        
        valid_losses = [l for l in all_final_losses if l is not None and l != float('inf')]

        if not valid_losses:
            print(f"The run failed or resulted in an invalid loss for Day {day_idx+1}.")
        else:
            best_loss = min(valid_losses)
            best_run_index = all_final_losses.index(best_loss)
            best_results = all_final_results[best_run_index]
            
            print(f"Best Run Loss: {best_loss} (after {steps_run} steps)")
            print(f"Final Parameters (Natural Scale): {best_results[0]}")
            print(f"Final Parameters (Phi Scale)    : {best_results[1]}")
            print(f"Final Parameters (Raw Log Scale): {best_results[2]}")

        end_time = time.time()
        print(f"\nTotal execution time: {end_time - start_time:.2f} seconds")

    except Exception as e:
        print(f"🔴 Day {day_idx+1} Failed: {e}")
        import traceback
        traceback.print_exc()
        continue


--- Processing Real Data: Day 11 (2024-07-11) ---

Pre-computing J-vector (Hamming taper)...
Pre-computing sample periodogram...
Pre-computing Hamming taper autocorrelation...
Data grid: 44x62, 8 time points. J-vector, Periodogram, Taper Autocorr on cpu.

============================== Initialization Run 1/1 ==============================
Starting with FIXED params (raw log-scale): [4.2042, 1.6348, 0.4721, -2.5562, 0.0218, -0.1689, -1.3984]
Starting optimization run 1 on device cpu (Hamming, 7-param ST kernel, L-BFGS)...
--- Step 1/15 ---
 Loss: -5.215840 | Max Grad: 7.458467e-03
  Params (Raw Log): log_phi1: 2.4045, log_phi2: 2.0656, log_phi3: 0.3280, log_phi4: -3.1575, advec_lat: 0.0073, advec_lon: -0.0254, log_nugget: -4.0986
--- Step 2/15 ---
 Loss: -13.872890 | Max Grad: 1.149569e-03
  Params (Raw Log): log_phi1: 2.4352, log_phi2: 2.0935, log_phi3: 0.3551, log_phi4: -3.0824, advec_lat: -0.0020, advec_lon: 0.0059, log_nugget: -8.1849
--- Step 3/15 ---
 Loss: -13.875208 | Max Grad: